# SK-DGCNN for MARS Skeleton Regression

This notebook trains the SK-DGCNN pose estimator from Zhang et al. (2026) to predict a Kinect skeleton from one synchronized radar point cloud frame.

We use the MARS dataset, whose features are `x, y, z, velocity, intensity`.

MARS Kinect files store `kinect_data_cropped/JointPositions` as one entry per synchronized frame. Per kinect entry we have: 19 Kinect joints times 3 coordinates = 57 values.

Dimension convention:

- B: batch size
- C: radar feature channels, set to 5
- N: radar points per frame, set to 25
- J: skeleton joints, set to 19
- D: skeleton coordinates per joint, set to 3

In [29]:
from functools import lru_cache
from pathlib import Path
import time

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import clear_output
from torch.utils.data import DataLoader, Dataset

In [ ]:
PROJECT_ROOT = Path('/Users/elo/Coding/radar_ml')
DATA_ROOT = PROJECT_ROOT / 'datasets' / 'MARS'
CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints' / 'MARS_SK_DGCNN'

ACTIVITY_CLASS_NAMES = (
    'Left_upper_limb_extension',
    'Right_upper_limb_extension',
    'Both_upper_limb_extension',
    'Left_front_lunge',
    'Right_front_lunge',
    'Squad',
    'Left_side_lunge',
    'Right_side_lunge',
    'Left_limb_extension',
    'Right_limb_extension',
)

SKELETON_JOINT_NAMES = (
    'SpineBase', 'SpineMid', 'Neck', 'Head',
    'ShoulderLeft', 'ElbowLeft', 'WristLeft', 'HandLeft',
    'ShoulderRight', 'ElbowRight', 'WristRight', 'HandRight',
    'HipLeft', 'KneeLeft', 'AnkleLeft',
    'HipRight', 'KneeRight', 'AnkleRight',
    'SpineShoulder',
)
SKELETON_JOINT_INDICES = (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 20)

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
CONFIG = {
    'radar_mat_key': 'radar_data_cropped',
    'kinect_mat_key': 'kinect_data_cropped',
    'frame_row': 0,
    'feature_names': ('x', 'y', 'z', 'velocity', 'intensity'),
    'feature_rows': (2, 3, 4, 5, 6),
    'feature_amount': 5,
    'num_points': 25,
    'validation_fraction': 0.2,
    'k': 20,
    'emb_dims': 512,
    'dropout': 0.4,
    'batch_size': 16,
    'test_batch_size': 64,
    'num_epochs': 150,
    'base_lr': 0.0005,
    'adam_momentum': 0.5,
    'axis_order': (2, 0, 1),
    'axis_phase_epochs': 50,
    'eca_gamma': 2,
    'eca_b': 1,
}

In [ ]:
print(f'Device: {DEVICE}')
print(f'Radar input shape: (B, C={CONFIG["feature_amount"]}, N={CONFIG["num_points"]})')
print(f'Skeleton target shape: (B, {len(SKELETON_JOINT_INDICES)} joints * 3 = {len(SKELETON_JOINT_INDICES) * 3})')
print(dict(zip(CONFIG['feature_names'], CONFIG['feature_rows'])))

# Data

In [33]:
def paired_kinect_file_path(radar_file_path):
    "Return the Kinect file paired with one radar file."
    return Path(str(radar_file_path).replace('/radar_data', '/kinect_data'))

In [34]:
def split_file_paths():
    "Split the prepared MARS Train files into train/validation and keep Test held out."
    train_files = []
    valid_files = []
    for class_name in ACTIVITY_CLASS_NAMES:
        file_paths = sorted((DATA_ROOT / 'Train').glob(f'subject*/{class_name}/radar_data*.mat'))
        val_count = max(1, int(len(file_paths) * CONFIG['validation_fraction']))
        val_count = min(val_count, len(file_paths) - 1)
        train_files.extend(str(path) for path in file_paths[:-val_count])
        valid_files.extend(str(path) for path in file_paths[-val_count:])
    test_files = [
        str(file_path)
        for class_name in ACTIVITY_CLASS_NAMES
        for file_path in sorted((DATA_ROOT / 'Test').glob(f'subject*/{class_name}/radar_data*.mat'))
    ]
    return train_files, valid_files, test_files

In [35]:
@lru_cache(maxsize=None)
def load_radar_frames_raw(file_path):
    "Load one MARS radar file as a list of raw point tensors."
    with h5py.File(file_path, 'r') as handle:
        radar = np.asarray(handle[CONFIG['radar_mat_key']])
    frame_ids = radar[CONFIG['frame_row']].astype(np.int64)
    features = radar[list(CONFIG['feature_rows'])].T.astype(np.float32)
    return [torch.from_numpy(features[frame_ids == frame_id]) for frame_id in np.unique(frame_ids)]

In [36]:
@lru_cache(maxsize=None)
def train_radar_normalization_stats():
    "Compute radar feature normalization stats over training subjects."
    train_files, _, _ = split_file_paths()
    points = [torch.cat(load_radar_frames_raw(file_path), dim=0) for file_path in train_files]
    points = torch.cat(points, dim=0)
    return points.mean(dim=0), points.std(dim=0).clamp_min(1e-6)

In [37]:
@lru_cache(maxsize=None)
def load_radar_frames(file_path):
    "Load one MARS radar file as normalized frame point clouds."
    mean, std = train_radar_normalization_stats()
    return [(frame - mean) / std for frame in load_radar_frames_raw(file_path)]

In [38]:
@lru_cache(maxsize=None)
def load_skeletons_raw(file_path):
    "Load one paired Kinect file as 19-joint skeleton vectors."
    kinect_file_path = paired_kinect_file_path(file_path)
    skeletons = []
    with h5py.File(kinect_file_path, 'r') as handle:
        refs = np.asarray(handle[CONFIG['kinect_mat_key']]['JointPositions']).ravel()
        for ref in refs:
            joints = np.asarray(handle[ref], dtype=np.float32)[0]
            joints = joints[:, SKELETON_JOINT_INDICES].T.reshape(-1)
            skeletons.append(torch.from_numpy(joints))
    return skeletons

In [39]:
@lru_cache(maxsize=None)
def train_skeleton_normalization_stats():
    "Compute skeleton coordinate normalization stats over training subjects."
    train_files, _, _ = split_file_paths()
    skeletons = [torch.stack(load_skeletons_raw(file_path), dim=0) for file_path in train_files]
    skeletons = torch.cat(skeletons, dim=0)
    return skeletons.mean(dim=0), skeletons.std(dim=0).clamp_min(1e-6)

In [40]:
@lru_cache(maxsize=None)
def load_skeletons(file_path):
    "Load one paired Kinect file as normalized skeleton vectors."
    mean, std = train_skeleton_normalization_stats()
    return [(skeleton - mean) / std for skeleton in load_skeletons_raw(file_path)]

In [41]:
def build_frame_index(file_paths):
    "Create synchronized radar-frame to skeleton-frame samples."
    samples = []
    for file_path in file_paths:
        frame_amount = min(len(load_radar_frames(file_path)), len(load_skeletons(file_path)))
        for frame_idx in range(frame_amount):
            samples.append((file_path, frame_idx))
    return samples

In [ ]:
def process_frame(points):
    "Pad or crop one radar frame to a fixed-size SK-DGCNN point cloud."
    output = torch.zeros(CONFIG['num_points'], CONFIG['feature_amount'], dtype=torch.float32)
    output[:min(len(points), CONFIG['num_points'])] = points[:CONFIG['num_points']]
    return output

In [43]:
class MARSSkeletonDataset(Dataset):
    "Dataset returning one radar point cloud and its synchronized Kinect skeleton."
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, frame_idx = self.samples[idx]
        point_cloud = process_frame(load_radar_frames(file_path)[frame_idx])
        skeleton = load_skeletons(file_path)[frame_idx]
        return point_cloud, skeleton

In [44]:
train_files, valid_files, test_files = split_file_paths()
train_samples = build_frame_index(train_files)
valid_samples = build_frame_index(valid_files)
test_samples = build_frame_index(test_files)
print(len(train_files), len(valid_files), len(test_files))
print(len(train_samples), len(valid_samples), len(test_samples))
print(train_samples[:3])

19 10 10
19336 11801 8885
[('/Users/elo/Coding/radar_ml/datasets/MARS/Left_upper_limb_extension/subject2/radar_data1.mat', 0), ('/Users/elo/Coding/radar_ml/datasets/MARS/Left_upper_limb_extension/subject2/radar_data1.mat', 1), ('/Users/elo/Coding/radar_ml/datasets/MARS/Left_upper_limb_extension/subject2/radar_data1.mat', 2)]


In [45]:
train_data = MARSSkeletonDataset(train_samples)
x, y = train_data[0]
print(x.shape, y.shape)
print(x[:3])
print(y[:9])

torch.Size([64, 4]) torch.Size([57])
tensor([[ 0.0814,  0.1987,  0.7981,  0.7095],
        [ 0.0814,  0.3784,  0.7504,  0.7095],
        [ 0.0814,  0.3014,  0.5594, -0.7170]])
tensor([-0.0312,  0.9267,  0.1995, -0.0611,  1.0304,  0.2681, -0.0877,  0.9527,
         0.3359])


# Architecture

In [46]:
def knn(x, k):
    "K nearest neighbor indexes from the original DGCNN implementation."
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x ** 2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2, 1)
    return pairwise_distance.topk(k=k, dim=-1)[1]

In [47]:
def get_graph_feature(x, k=20, idx=None):
    "Build EdgeConv features as concat(neighbor - center, center)."
    batch_size = x.size(0)
    num_points = x.size(2)
    x = x.view(batch_size, -1, num_points)
    if idx is None:
        idx = knn(x, k=k)
    idx_base = torch.arange(0, batch_size, device=x.device).view(-1, 1, 1) * num_points
    idx = (idx + idx_base).view(-1)
    _, num_dims, _ = x.size()
    x = x.transpose(2, 1).contiguous()
    feature = x.view(batch_size * num_points, -1)[idx, :]
    feature = feature.view(batch_size, num_points, k, num_dims)
    x = x.view(batch_size, num_points, 1, num_dims).repeat(1, 1, k, 1)
    return torch.cat((feature - x, x), dim=3).permute(0, 3, 1, 2).contiguous()

In [ ]:
class ECALayer(nn.Module):
    "Efficient channel attention for SK-DGCNN feature maps."
    def __init__(self, channels):
        super().__init__()
        kernel_size = int(abs(np.log2(channels) / CONFIG['eca_gamma'] + CONFIG['eca_b']))
        kernel_size = kernel_size if kernel_size % 2 else kernel_size + 1
        self.conv = nn.Conv1d(1, 1, kernel_size=kernel_size, padding=(kernel_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = x.mean(dim=tuple(range(2, x.dim()))).unsqueeze(1)
        y = self.conv(y).squeeze(1).view(x.size(0), x.size(1), *([1] * (x.dim() - 2)))
        return x * self.sigmoid(y)

In [ ]:
class DGCNNSkeletonRegressor(nn.Module):
    "SK-DGCNN regressor predicting a flattened 19-joint skeleton."
    def __init__(self):
        super().__init__()
        self.k = CONFIG['k']
        self.joints = len(SKELETON_JOINT_INDICES)
        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm1d(CONFIG['emb_dims'])
        self.bn5 = nn.BatchNorm1d(512)
        self.bn6 = nn.BatchNorm1d(self.joints * 128)
        self.conv1 = nn.Sequential(nn.Conv2d(CONFIG['feature_amount'] * 2, 64, kernel_size=1, bias=False), self.bn1, nn.LeakyReLU(negative_slope=0.2))
        self.conv2 = nn.Sequential(nn.Conv2d(64 * 2, 64, kernel_size=1, bias=False), self.bn2, nn.LeakyReLU(negative_slope=0.2))
        self.conv3 = nn.Sequential(nn.Conv2d(64 * 2, 128, kernel_size=1, bias=False), self.bn3, nn.LeakyReLU(negative_slope=0.2))
        self.conv4 = nn.Sequential(nn.Conv1d(256, CONFIG['emb_dims'], kernel_size=1, bias=False), self.bn4, nn.LeakyReLU(negative_slope=0.2))
        self.eca1 = ECALayer(64)
        self.eca2 = ECALayer(64)
        self.eca3 = ECALayer(128)
        self.eca4 = ECALayer(CONFIG['emb_dims'])
        self.linear1 = nn.Linear(CONFIG['emb_dims'] * 2, 512, bias=False)
        self.dp1 = nn.Dropout(p=CONFIG['dropout'])
        self.linear2 = nn.Linear(512, self.joints * 128, bias=False)
        self.dp2 = nn.Dropout(p=CONFIG['dropout'])
        self.x_head = nn.Linear(128, 1)
        self.y_head = nn.Linear(128, 1)
        self.z_head = nn.Linear(128, 1)

    def forward(self, x):
        batch_size = x.size(0)
        x = self.eca1(self.conv1(get_graph_feature(x, k=self.k)))
        x1 = x.max(dim=-1, keepdim=False)[0]
        x = self.eca2(self.conv2(get_graph_feature(x1, k=self.k)))
        x2 = x.max(dim=-1, keepdim=False)[0]
        x = self.eca3(self.conv3(get_graph_feature(x2, k=self.k)))
        x3 = x.max(dim=-1, keepdim=False)[0]
        x = torch.cat((x1, x2, x3), dim=1)
        x = self.eca4(self.conv4(x))
        x = torch.cat((F.adaptive_max_pool1d(x, 1).view(batch_size, -1), F.adaptive_avg_pool1d(x, 1).view(batch_size, -1)), 1)
        x = self.dp1(F.leaky_relu(self.bn5(self.linear1(x)), negative_slope=0.2))
        x = self.dp2(F.leaky_relu(self.bn6(self.linear2(x)), negative_slope=0.2))
        x = x.view(batch_size, self.joints, 128)
        x = torch.cat((self.x_head(x), self.y_head(x), self.z_head(x)), dim=2)
        return x.reshape(batch_size, -1)

In [ ]:
model = DGCNNSkeletonRegressor()
train_loader = DataLoader(train_data, batch_size=CONFIG['batch_size'], shuffle=True, drop_last=True)
sample_batch, sample_skeletons = next(iter(train_loader))
logits = model(sample_batch.permute(0, 2, 1))
print(sample_batch.shape)
print(sample_batch.permute(0, 2, 1).shape)
print(logits.shape, sample_skeletons.shape)

# Training

In [50]:
def mean_per_joint_position_error(predictions, targets):
    "Compute MPJPE in Kinect coordinate units."
    mean, std = train_skeleton_normalization_stats()
    predictions = (predictions * std + mean).view(-1, len(SKELETON_JOINT_INDICES), 3)
    targets = (targets * std + mean).view(-1, len(SKELETON_JOINT_INDICES), 3)
    return torch.linalg.vector_norm(predictions - targets, dim=2).mean().item()

In [ ]:
def axis_position_loss(predictions, targets, axis, epoch):
    "Compute the paper's dynamic MSE plus KLDiv loss for one coordinate axis."
    predictions = predictions.view(-1, len(SKELETON_JOINT_INDICES), 3)[:, :, axis]
    targets = targets.view(-1, len(SKELETON_JOINT_INDICES), 3)[:, :, axis]
    alpha_kl = 0.05 * (epoch // 20)
    alpha_mse = 1 - alpha_kl
    mse = F.mse_loss(predictions, targets)
    kldiv = F.kl_div(F.log_softmax(predictions, dim=1), F.softmax(targets, dim=1), reduction='batchmean')
    return alpha_mse * mse + alpha_kl * kldiv

In [ ]:
def skeleton_loss(predictions, targets, epoch):
    "Apply sequential multitask loss in z, x, y order."
    phase = min(epoch // CONFIG['axis_phase_epochs'], len(CONFIG['axis_order']) - 1)
    losses = [axis_position_loss(predictions, targets, axis, epoch) for axis in CONFIG['axis_order'][:phase + 1]]
    return sum(losses)

In [51]:
def create_dataloaders():
    "Create train, validation, and test dataloaders."
    train_loader = DataLoader(MARSSkeletonDataset(train_samples), batch_size=CONFIG['batch_size'], shuffle=True, drop_last=True)
    valid_loader = DataLoader(MARSSkeletonDataset(valid_samples), batch_size=CONFIG['test_batch_size'], shuffle=False, drop_last=False)
    test_loader = DataLoader(MARSSkeletonDataset(test_samples), batch_size=CONFIG['test_batch_size'], shuffle=False, drop_last=False)
    return train_loader, valid_loader, test_loader

In [ ]:
def run_epoch(model, dataloader, epoch=0, optimizer=None):
    "Run one training or evaluation epoch."
    model.train(optimizer is not None)
    losses = []
    grad_norms = []
    all_predictions = []
    all_targets = []
    for data, target in dataloader:
        data = data.to(DEVICE).permute(0, 2, 1)
        target = target.to(DEVICE)
        if optimizer is not None:
            optimizer.zero_grad()
        predictions = model(data)
        loss = skeleton_loss(predictions, target, epoch)
        if optimizer is not None:
            loss.backward()
            grad_norm = torch.sqrt(sum(p.grad.detach().pow(2).sum() for p in model.parameters() if p.grad is not None)).item()
            optimizer.step()
            grad_norms.append(grad_norm)
        losses.append(loss.item())
        all_predictions.append(predictions.detach().cpu())
        all_targets.append(target.detach().cpu())
    all_predictions = torch.cat(all_predictions)
    all_targets = torch.cat(all_targets)
    return losses, grad_norms, mean_per_joint_position_error(all_predictions, all_targets)

In [ ]:
def plot(train_losses, valid_losses, all_grad_norms, model_param_count, num_epochs, train_epoch_sizes, valid_epoch_sizes, epoch_means):
    "Plot the losses and the gradient norm over the training process."
    x_train = [e + i / s for e, s in enumerate(train_epoch_sizes) for i in range(s)]
    x_valid = [e + i / s for e, s in enumerate(valid_epoch_sizes) for i in range(s)]
    epoch_mean_valid_losses = [m for s, m in zip(valid_epoch_sizes, epoch_means) for _ in range(s)]

    plt.figure(figsize=(10, 10))
    ax1 = plt.subplot(2, 1, 1)
    w = max(1, int((len(train_losses) / num_epochs) * 0.1))
    train_loss_rolling = pd.Series(train_losses).rolling(window=w, center=True).mean()
    plt.plot(x_train, train_losses, label="Training Loss"); plt.plot(x_train, train_loss_rolling, label="Training Loss (Avg)", color="darkblue", linewidth=1); plt.plot(x_valid, valid_losses, label="Validation Loss", alpha=0.7); plt.plot(x_valid, epoch_mean_valid_losses, label="Val Loss (Avg)", color="red", linewidth=1, linestyle="--")
    plt.xlabel("Epochs"); plt.ylabel("Loss"); plt.legend(); plt.title("Training and Validation Loss Over Epochs")
    ax1.set_xlim(0, num_epochs); ax1.set_xticks(range(num_epochs + 1))
    loss_values = pd.Series(train_losses + valid_losses + epoch_means).dropna()
    q1, q3 = loss_values.quantile(0.25), loss_values.quantile(0.75)
    bottom, top = max(0, q1 - 1.5 * (q3 - q1)), q3 + 1.5 * (q3 - q1)
    if top < loss_values.max(): ax1.set_ylim(bottom, top + max((top - bottom) * 0.05, 1e-6))
    y_min, y_max = ax1.get_ylim()
    best_mean_val = float("inf")
    for i, mean_val in enumerate(epoch_means):
        if mean_val <= best_mean_val and y_min <= mean_val <= y_max: ax1.text(i + 0.5, mean_val, f"{mean_val:.3g}", color="red", ha="center", va="bottom", fontweight="medium", clip_on=True)
        best_mean_val = min(best_mean_val, mean_val)

    ax = plt.subplot(2, 1, 2)
    plt.plot(x_train, all_grad_norms, label="Gradient Norm", color="orange"); plt.xlabel("Epochs"); plt.ylabel("Norm"); plt.legend(); plt.title("Gradient Norm over Epochs")
    ax.set_xlim(0, num_epochs); ax.set_xticks(range(num_epochs + 1))
    grad_values = pd.Series(all_grad_norms).dropna()
    q1, q3 = grad_values.quantile(0.25), grad_values.quantile(0.75)
    bottom, top = max(0, q1 - 1.5 * (q3 - q1)), q3 + 1.5 * (q3 - q1)
    if top < grad_values.max(): ax.set_ylim(bottom, top + max((top - bottom) * 0.05, 1e-6))
    ax.text(0.99, 0.02, f"Params count: {model_param_count:,}", transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color="dimgray")

    plt.subplots_adjust(hspace=0.35); plt.show()

In [ ]:
def train_model(num_epochs=CONFIG['num_epochs']):
    "Launch the training process of the SK-DGCNN skeleton regressor."
    print('Training process starting...', flush=True)
    CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
    model = DGCNNSkeletonRegressor().to(DEVICE)
    model_param_count = sum(p.numel() for p in model.parameters())
    train_loader, valid_loader, _ = create_dataloaders()
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['base_lr'], betas=(CONFIG['adam_momentum'], 0.999), weight_decay=0)
    train_losses, valid_losses, all_grad_norms = [], [], []
    train_epoch_sizes, valid_epoch_sizes, epoch_means = [], [], []
    best_valid_loss = float('inf')

    for epoch in range(num_epochs):
        start = time.time()
        axis_names = [CONFIG['feature_names'][i] for i in CONFIG['axis_order'][:min(epoch // CONFIG['axis_phase_epochs'], 2) + 1]]
        print(f'Epoch n°{epoch} Training axes {axis_names} ====', flush=True)
        loss_list, grad_norms, train_mpjpe = run_epoch(model, train_loader, epoch, optimizer)
        train_losses.extend(loss_list)
        train_epoch_sizes.append(len(loss_list))
        all_grad_norms.extend(grad_norms)

        file_path = CHECKPOINT_ROOT / f'checkpoint_{epoch:02d}.pt'
        torch.save(model.state_dict(), file_path)

        print(f'Epoch n°{epoch} Validation ====', flush=True)
        with torch.no_grad():
            loss_list, _, valid_mpjpe = run_epoch(model, valid_loader, epoch)
        valid_losses.extend(loss_list)
        valid_epoch_sizes.append(len(loss_list))

        mean_val = sum(loss_list) / len(loss_list)
        epoch_means.append(mean_val)
        if mean_val <= best_valid_loss:
            best_valid_loss = mean_val
            torch.save(model.state_dict(), CHECKPOINT_ROOT / 'model.pt')
        clear_output(wait=True)
        print('Training process starting...', flush=True)
        print(f'Epoch n°{epoch} complete | Train MPJPE: {train_mpjpe:.4f} | Validation loss: {mean_val:.4f} | Validation MPJPE: {valid_mpjpe:.4f} | Time: {time.time() - start:.1f}s', flush=True)
        plot(train_losses, valid_losses, all_grad_norms, model_param_count, num_epochs, train_epoch_sizes, valid_epoch_sizes, epoch_means)

    return model

In [ ]:
model = train_model()

# Inference

In [ ]:
def evaluate_validation_checkpoint(checkpoint_path):
    "Score a checkpoint on validation only."
    device = torch.device(DEVICE)
    model = DGCNNSkeletonRegressor().to(device)
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    _, valid_loader, _ = create_dataloaders()
    with torch.no_grad():
        losses, _, mpjpe = run_epoch(model, valid_loader, CONFIG['num_epochs'] - 1)
    return sum(losses) / len(losses), mpjpe

In [ ]:
checkpoint_paths = sorted(CHECKPOINT_ROOT.glob('checkpoint_*.pt'), key=lambda path: int(path.stem.split('_')[-1]))
metrics = [evaluate_validation_checkpoint(path) for path in checkpoint_paths]
checkpoint_labels = [path.stem for path in checkpoint_paths]
validation_losses = [metric[0] for metric in metrics]
validation_mpjpes = [metric[1] for metric in metrics]

plt.figure(figsize=(10, 5))
loss_line, = plt.plot(checkpoint_labels, validation_losses, marker='o', label='Validation loss')
mpjpe_line, = plt.plot(checkpoint_labels, validation_mpjpes, marker='o', label='Validation MPJPE')

best_loss = float('inf')
for idx, score in enumerate(validation_losses):
    if score < best_loss:
        plt.text(idx, score, f'{score:.3f}', ha='center', va='bottom', color=loss_line.get_color())
        best_loss = score

best_mpjpe = float('inf')
for idx, score in enumerate(validation_mpjpes):
    if score < best_mpjpe:
        plt.text(idx, score, f'{score:.3f}', ha='center', va='bottom', color=mpjpe_line.get_color())
        best_mpjpe = score

plt.xticks(rotation=45)
plt.ylabel('Score')
plt.title('Validation checkpoint performance')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_idx = min(range(len(validation_mpjpes)), key=lambda idx: validation_mpjpes[idx])
best_checkpoint_path = checkpoint_paths[best_idx]
device = torch.device(DEVICE)
model = DGCNNSkeletonRegressor().to(device)
state_dict = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

_, _, test_loader = create_dataloaders()
with torch.no_grad():
    test_losses, _, test_mpjpe = run_epoch(model, test_loader, CONFIG['num_epochs'] - 1)
test_loss = sum(test_losses) / len(test_losses)
print(f'Best validation checkpoint: {best_checkpoint_path.name}')
print(f'Validation MPJPE: {validation_mpjpes[best_idx]:.6f}')
print(f'Validation loss: {validation_losses[best_idx]:.6f}')
print(f'Test loss: {test_loss:.6f}')
print(f'Test MPJPE: {test_mpjpe:.6f}')